# A simple example of multiple dispatch: animals

From this very good blog post [First Impressions of Julia from an R User](https://mdneuzerling.com/post/first-impressions-of-julia-from-an-r-user/).

See [here](https://erikexplores.substack.com/p/what-makes-julia-unique) for another reference and there for a [Pokemon TP](https://gdalle.github.io/JuliaComputationSolutions/hw1a_solutions.html) both on multiple dispatch.

In [1]:
abstract type Animal end

In [2]:
struct Cat <: Animal
    Name::String 
    Age::Int
end

struct Dog <: Animal
    Name::String 
    Age::Int
end

In [3]:
interaction(x::Cat, y::Cat) = "meow"

interaction (generic function with 1 method)

In [4]:
interaction(x::Dog, y::Dog) = "sniff"

interaction (generic function with 2 methods)

In [5]:
interaction(x::Cat, y::Dog) = "growl"

interaction (generic function with 3 methods)

In [6]:
interaction(x::Dog, y::Cat) = interaction(y, x)

interaction (generic function with 4 methods)

In [7]:
luna = Cat("Luna", 1)
pip = Cat("Pip", 5)
hudson = Dog("Hudson", 4)
phoebe = Dog("Phoebe", 1)

Dog("Phoebe", 1)

In [8]:
interaction(luna, pip)

"meow"

In [9]:
interaction(pip, luna)

"meow"

In [10]:
interaction(hudson, phoebe)

"sniff"

## Going up the type hierarchy

In [11]:
struct Gazelle <: Animal
    Name::String 
    Age::Int
end

In [12]:
bob = Gazelle("Bob", 2);
interaction(x::Animal, y::Animal) = "flee";

This is where it would not work (easily) with classes!

In [13]:
interaction(hudson, bob)

"flee"

In [14]:
@which interaction(hudson, bob)

interaction(x::Animal, y::Animal)
     @ Main c:\Users\metivier\Dropbox\PC (2)\Documents\dev\MyJuliaIntroDocs\notebooks\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X20sZmlsZQ==.jl:2

# A useful example: One Hot Vector
Vector of length $n$ with only one nonzero element.
$(0,0,\cdots, 1, \cdots, 0,0)$
Commonly used in Machine Learning

In [15]:
using BenchmarkTools

Overload the `Base` functions `size`, `getindex`, `*`

In [16]:
import Base: size, getindex, *

In [17]:
struct OneHotVector <: AbstractVector{Bool}
    len::Int
    ind::Int
end

`size` and `getindex` are all you need to define a vector object

In [18]:
size(v::OneHotVector) = (v.len, )

size (generic function with 88 methods)

In [20]:
getindex(v::OneHotVector, i::Integer) = i == v.ind ? 1 : 0

getindex (generic function with 206 methods)

Define a `Matrix` named `A` of `rand` of size $10^4\times 10^4$

In [21]:
A = rand(10^4, 10^4)

10000×10000 Matrix{Float64}:
 0.963128   0.3579    0.015288   0.429355   …  0.322772  0.821153   0.198544
 0.0367084  0.131771  0.19416    0.344206      0.2929    0.793191   0.534539
 0.769039   0.769312  0.57653    0.321025      0.399248  0.931139   0.558764
 0.513517   0.207988  0.129289   0.785254      0.84022   0.741167   0.909843
 0.975374   0.744676  0.906732   0.841663      0.612695  0.887687   0.67812
 0.66025    0.821254  0.701985   0.643265   …  0.920533  0.895778   0.63395
 0.0223434  0.513874  0.879068   0.577685      0.685457  0.0501986  0.703804
 0.165541   0.411433  0.127421   0.90541       0.74914   0.524031   0.310295
 0.0690079  0.747006  0.656501   0.821927      0.611799  0.370251   0.193999
 0.761587   0.681157  0.165723   0.351962      0.383993  0.162367   0.854088
 ⋮                                          ⋱                       
 0.248934   0.193694  0.0845728  0.877116      0.247855  0.694459   0.163806
 0.596515   0.949323  0.608944   0.0461287     0.849725  

Define a `OneHotVector` named `v` of length $10^4$ with nonzero element at index $1000$

In [22]:
v = OneHotVector(10^4, 1000)

10000-element OneHotVector:
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 ⋮
 0
 0
 0
 0
 0
 0
 0
 0
 0

In [23]:
@which A*v

*(A::AbstractMatrix{T}, x::AbstractVector{S}) where {T, S}
     @ LinearAlgebra C:\Users\metivier\.julia\juliaup\julia-1.11.2+0.x64.w64.mingw32\share\julia\stdlib\v1.11\LinearAlgebra\src\matmul.jl:58

In [24]:
@btime A*v

  41.376 ms (3 allocations: 78.19 KiB)


10000-element Vector{Float64}:
 0.25025730896477383
 0.8265131249885858
 0.025195855929133093
 0.7627130476470503
 0.6818833529058055
 0.34091663574438547
 0.6615756242048698
 0.41430386654569296
 0.4840101416977939
 0.9122976464852238
 ⋮
 0.27978522097918324
 0.45577851604814246
 0.6967091038934528
 0.5187546464880788
 0.9510573817711486
 0.6256114625220106
 0.9408342818863769
 0.550015391596959
 0.8666737424219615

In [25]:
*(a::AbstractMatrix,v::OneHotVector) = a[:, v.ind]

* (generic function with 225 methods)

In [26]:
@btime A*v

  5.667 μs (3 allocations: 78.19 KiB)


10000-element Vector{Float64}:
 0.25025730896477383
 0.8265131249885858
 0.025195855929133093
 0.7627130476470503
 0.6818833529058055
 0.34091663574438547
 0.6615756242048698
 0.41430386654569296
 0.4840101416977939
 0.9122976464852238
 ⋮
 0.27978522097918324
 0.45577851604814246
 0.6967091038934528
 0.5187546464880788
 0.9510573817711486
 0.6256114625220106
 0.9408342818863769
 0.550015391596959
 0.8666737424219615

In [27]:
@which A*v

*(a::AbstractMatrix, v::OneHotVector)
     @ Main c:\Users\metivier\Dropbox\PC (2)\Documents\dev\MyJuliaIntroDocs\notebooks\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X45sZmlsZQ==.jl:1